In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *
from delta.tables import DeltaTable

df_old = spark.read.table("db_catalog.src.regions")
df_old.display()

region_id,region,_rescued_data
R01,East,null
R02,West,null
R03,North,null
R04,South,null


In [0]:
df_old= df_old.drop("_rescued_data")
df_old.display()

region_id,region
R01,East
R02,West
R03,North
R04,South


In [0]:
init_load_flag  = not spark.catalog.tableExists("db_catalog.gold.dim_regions")
print(init_load_flag)

False


In [0]:
if init_load_flag:
    df_dim_regions = (
        df_old
        .withColumn("dim_region_key", monotonically_increasing_id() + lit(1))
        .withColumn("create_date", current_timestamp())
        .withColumn("update_date", current_timestamp())
        .withColumn("current_flag", lit(True))
    )

    df_dim_regions = df_dim_regions.select("dim_region_key", "region_id", "region", "create_date", "update_date", "current_flag")
    df_dim_regions.write.format("delta").mode("overwrite").save("abfss://source@dbproject.dfs.core.windows.net/gold")

    spark.sql(f"CREATE TABLE IF NOT EXISTS db_catalog.gold.dim_regions USING DELTA LOCATION 'abfss://source@dbproject.dfs.core.windows.net/gold'")

else:

    df_regions_gold = spark.table("db_catalog.gold.dim_regions")
    max_dim_region_key = (df_regions_gold.agg(max("dim_region_key").alias("max_key")).collect()[0]["max_key"])
    df_region_changes = (df_old.alias("source").join(df_regions_gold.alias("gold"), col("source.region_id") == col("gold.region_id"), "left"))
    df_region_changes = df_region_changes.withColumn("is_changed", ~col("source.region").eqNullSafe(col("gold.region")))
    df_region_changes = df_region_changes.withColumn("is_new", col("gold.region_id").isNull())

    window_spec = Window.orderBy("source.region_id")

    df_new_regions = (df_region_changes.filter(col("is_new") == True).withColumn("dim_region_key", row_number().over(window_spec) + max_dim_region_key).select(col("source.region_id").alias("region_id"), col("source.region").alias("region"), col("dim_region_key")).withColumn("is_changed", lit(False)))

    df_existing_regions = (df_region_changes.filter(col("is_new") == False).select(col("source.region_id").alias("region_id"), col("source.region").alias("region"), col("gold.dim_region_key").alias("dim_region_key"), col("is_changed")))

    df_region_merge = (df_existing_regions.unionByName(df_new_regions).withColumn("update_date", current_timestamp()))

    delta_regions = DeltaTable.forName(spark, "db_catalog.gold.dim_regions")
    (delta_regions.alias("gold").merge(df_region_merge.alias("source"), "gold.region_id = source.region_id").whenMatchedUpdate(condition="source.is_changed = true", set={"region": "source.region", "update_date": "source.update_date", "current_flag": "true"}).whenNotMatchedInsert(values={"dim_region_key": "source.dim_region_key", "region_id": "source.region_id", "region": "source.region",
    "create_date": "source.update_date", "update_date": "source.update_date", "current_flag": "true"}).execute()
    )

    print("SCD Type 1 merge completed")




/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


SCD Type 1 merge completed


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
df_regions_gold = spark.table(
    "db_catalog.gold.dim_regions"
)

df_regions_gold.display()

dim_region_key,region_id,region,create_date,update_date,current_flag
1,R01,East,2026-09-08T13:24:40.620Z,2026-09-08T13:24:40.620Z,true
2,R02,West,2026-09-08T13:24:40.620Z,2026-09-08T13:24:40.620Z,true
3,R03,North,2026-09-08T13:24:40.620Z,2026-09-08T13:24:40.620Z,true
4,R04,South,2026-09-08T13:24:40.620Z,2026-09-08T13:24:40.620Z,true
